<br>
<div style="text-align:center">
    <img src="https://raw.githubusercontent.com/cassidythilton/portfolio/main/po001_mrm/src/headerImage.png?token=GHSAT0AAAAAACH3RUYRNC2MYO3NTJ4AATMEZISABGA">
<br>
</div>   



<h1  style="text-align:center;font-size:.95em;font-style:italic;font-weight:200;line-height:2.5;">This notebook is a supplement to the primary Monitoring Solution notebook. It is designed to automate the generation of Model Risk Management (MRM) reports in the financial sector, particularly for fraud detection models. By leveraging the metrics extracted from the Project and Policies in the primary notebook, we incorporate these measures into our ongoing model performance monitoring plan. <br> Our goal is to create a fraud detection model that is compliant with the SR Letter 11-7 guidelines, while ensuring it remains robust, accurate, and stable in detecting and preventing fraudulent transactions. The generated MRM report is output as an HTML document, which not only provides a convenient format for viewing and sharing, but also allows for easy integration into other systems if necessary.  </h1>


<br>



>1. <span style="color:#4c4c4c;font-size:.95em;font-weight:900;line-height:2.5;">&#9679;&nbsp;</span> [Specify Credentials](#1.-Specify-Credentials)  <span style="color:#4c4c4c;font-size:.75em;font-weight:300;">[user inputs required]</span>
>1. <span style="color:#55E7B2;font-size:.95em;font-weight:900;line-height:2.5;">&#9679;&nbsp;</span> [Read Variables](#2.-Read-Variables) <span style="color:#4c4c4c;font-size:.75em;font-weight:300;">[no user input or changes required]</span>
>1. <span style="color:#55E7B2;font-size:.95em;font-weight:900;line-height:2.5;">&#9679;&nbsp;</span> [Metrics & MRM Report Generation](#3.-Metrics-&-MRM-Report-Generation) <span style="color:#4c4c4c;font-size:.75em;font-weight:300;">[no user input or changes required]</span>

<span style="color:#f3f3f3f;font-size:.95em;font-weight:300;">⸺</span> 
<br>
<span style="color:#4c4c4c;font-size:.75em;font-weight:900;">&#9679;</span>
 <span style="color:#f3f3f3f;font-size:.95em;font-weight:300;"> requires input</span> 
<br>
<span style="color:#7AB07B;font-size:.75em;font-weight:900;">&#9679;</span>
 <span style="color:#f3f3f3f;font-size:.95em;font-weight:300;"> no user input or changes required</span> 

<br>


<div style="border: 2px solid #525F95 ; padding: 10px 10px; display: inline-block; background-color: #08101D;">
    <h1 style="text-align:left;font-size:.85em;font-style:italic;font-weight:900;color:#FFFFFF; margin: 0em 0;line-height:2.5;">drop your new file(s) at left</h1>
    <p style="margin: 0em 0;"><span style="font-size:.7em;color:#4c4c4c;line-height:2.5;"><code>template.html</code> and <code>variables.json</code> are accepted</span></p>
    <h1 style="text-align:left;font-size:1.25em;font-weight:700;color:#ffffff; margin: 0em 0;line-height:2.5;"> <<<<<<<< </h1>


</div>



In [ ]:
from helper import *
from helperMRM import *

## 1. Specify Credentials

<span style="color:#8c8c8c;font-size:.8em;font-weight:900;">&#9679;</span>
<span style="font-size:.8em;font-weight:300;"> [user inputs required]</span> 
<p><strong>Purpose:</strong> <span style="font-size:.8em;">The following cells are designed to capture user inputs necessary for the setup and configuration of various SDK functions as well as Snowflake and Databricks.</span></p>
<p><strong>Capabilities:</strong> <span style="font-size:.8em;">The cells allow for user interactivity, accepting user inputs including cluster (of the environment) User Name and Password. Additionally, inputs can be accepted for a specific project_name, additional metrics, and custom date ranges. There is also a list where multiple files can be recorded in order for multiple reports to be generated. All of the aforementioned inputs will serve as variables for the subsequent notebook cells. It functions as an interface for the user to log in and set variables that will influence the operations of the other cells in the notebook.</span></p>
<p><strong>Outputs:</strong> <span style="font-size:.8em;">The output of the cells are primarily the set of user-defined variables that will be used in the subsequent cells. It doesn't produce a tangible output in itself, but sets the stage for the other operations in the notebook.</span></p>
<p><strong>Additional Notes:</strong> <span style="font-size:.8em;">
<ul>
<li>  <span  style="font-size:.8em;"><strong>cluster</strong>: e.g. acme.oc.site.</span></li>
<li>  <span  style="font-size:.8em;"><strong>username</strong>: user name.</span></li>
<li>  <span  style="font-size:.8em;"><strong>password</strong>: password.</span></li>
<li>  <span  style="font-size:.8em;"><strong>vars_file_name</strong>: Name of the variables file. If set to <code>None</code>, will default to variables name from Monitoring Solution notebook.</span></li>
<li>  <span  style="font-size:.8em;"><strong>files</strong>: List of all files to be generated. Ensure templates have proper variables.</span></li>
</ul>    
<br>
Of note, the <code>abstract</code> variable allows for abstracting the metrics sent to chatGPT in multiple ways depending on the level of confidentiality and abstraction needed. The options for abstraction include:     
<ul>
    <li>  <span style="font-size:.7em;"> Actual [<code>actual</code>]: Returns the data as is, without any abstraction. No modifications or transformations are applied. </span></li>
    <li>  <span style="font-size:.7em;">Scaling [<code>scaling</code>]: Scales the values of the data between a specified range, such as 0 to 1. Retains the variation in the data but loses the original units. Min-Max scaling is an example of this. </span></li>
    <li>  <span style="font-size:.7em;">Noise addition [<code>noise</code>]: Adds random noise to the data, distorting the exact values while preserving the underlying pattern. Useful for creating synthetic datasets or introducing variability to data. </span></li>
    <li>  <span style="font-size:.7em;">Natural Language Processing [<code>language</code>]: Uses several ways to abstract data using NLP techniques: </span></li>
    <UL>
    <li>    <span style="font-size:.7em;">Summarization: Instead of providing raw data, summarizes the main trends or changes in the data using natural language descriptions. </span></li>
    <li>    <span style="font-size:.7em;">Categorical Interpretation: Describes the changes in metrics using predefined categories instead of raw numbers. Categories like "negligible," "small," "medium," or "large" can be used based on percentage changes. </span></li>
    <li>    <span style="font-size:.7em;">Generalized Statements: Offers generalized statements that give a sense of the data's performance without revealing specific numbers. Provides high-level descriptions such as improvements, decreases, consistency, or exceptions in the data. </span></li>
    </UL>
    <li>  <span style="font-size:.7em;">No summary [<code>None</code>]: No summary included. </span></li>
</ul>    
    </span></p>

In [3]:
cluster = get_from_sys("CLUSTER") #e.g. main1683140183.acme.cloud
username = get_from_sys("USERNAME") #acme UI login user
password = get_from_sys("PASSWORD") #acme UI login pw
abstract = 'actual' #The method for abstracting metrics sent to chatGPT 

file_config = {    
    "files": ['mrm_section_10_template_descr_demo'],
}

## 2. Read Variables
<span style="color:#7AB07B;font-size:.8em;font-weight:900;">&#9679;</span>
<span style="color:#4c4c4c;font-size:.8em;font-weight:300;"> [no input required]</span> 
<p><strong>Purpose:</strong> <span style="font-size:.8em;">To prepare and set up the necessary variables for subsequent analytical processes.</span></p>
<p><strong>Capabilities:</strong> <span style="font-size:.8em;">This cell is capable of loading saved variables from the main ACME Monitoring Solution notebook, such as model name, model version, and pertinent policy names.</span></p>
<p><strong>Outputs:</strong> <span style="font-size:.8em;">The output is a set of initialized variables that are used in the subsequent cells to perform various operations.</span></p>
<p><strong>Additional Notes:</strong> <span style="font-size:.8em;">These variables are crucial for querying the drift metrics in the next cell and for correctly identifying the model and its associated policies.</span></p>

In [15]:
token = login(username, password, cluster)
d = reload_yaml()
d["policy_name"] = getPolicies(d)
policy_name = d["policy_name"]
pR(d)

   ...cassidyh logged in successfully to dev1691613641.acme.info
   ...reloading vars

{
       "WEBSERVICES_URL": "https://webservices.dev1691613641.acme.info",
       "allcolumns": [
              "customer_id",
              "terminal_id",
              "tx_amount",
              "tx_fraud",
              "tx_fraud_scenario",
              "tx_during_weekend",
              "tx_during_night",
              "customer_id_nb_tx_1day_window",
              "customer_id_avg_amount_1day_window",
              "customer_id_nb_tx_7day_window",
              "customer_id_avg_amount_7day_window",
              "customer_id_nb_tx_30day_window",
              "customer_id_avg_amount_30day_window",
              "terminal_id_nb_tx_1day_window",
              "terminal_id_risk_1day_window",
              "terminal_id_nb_tx_7day_window",
              "terminal_id_risk_7day_window",
              "terminal_id_nb_tx_30day_window",
              "terminal_id_risk_30day_window",
              "tx_fra

## 3. Metrics Calculation

<span style="color:#7AB07B;font-size:.8em;font-weight:900;">&#9679;</span>
<span style="color:#4c4c4c;font-size:.8em;font-weight:300;"> [no input required]</span> 

<p><strong>Purpose:</strong> <span style="font-size:.8em;">To calculate and explain changes in key performance and drift-related metrics of the model over time and generate an MRM (Model Risk Management Report).</span></p>
<p><strong>Capabilities:</strong> <span style="font-size:.8em;">This cell can calculate various period over period deltas for all major model metrics available in the ACME platform. This includes standard metrics such as F1 score, accuracy, balanced accuracy, precision, and recall, drift related metrics such as macro and micro related weighted drift, critical and warning drift related exceptions, as well as other more proprietary metrics.</span></p>
<p><strong>Outputs:</strong> <span style="font-size:.8em;">The output is a comprehensive set of calculated deltas for the various metrics.</span></p>
<p><strong>Additional Notes:</strong> <span style="font-size:.8em;">This cell leverages Open AI's chatGPT. The performance metrics can be structured in multiple ways depending on the level of confidentiality and abstraction needed.
    </span></p>

In [23]:
generateMRMeval(token
    ,cluster
    ,d
    ,file_config
    ,abstract=abstract
    )

...parsing mrm/mrm_section_10_template_descr_demo.html
...extracted 2 requests

   ...analyzing request 1
   ...inquiring about appropriate task path
   ...querying ACME model performance endpoint
   ...generated image 1
   ...using scaled data for summarization

   ...analyzing request 2
   ...inquiring about appropriate task path
   ...querying ACME model performance endpoint
   ...added metric: metrics.False.fdr
   ...added metric: metrics.False.nlr
   ...gini coefficient not supported
   ...custom function found:
         def cost_based_fscore(precision, recall, alpha=0.5, beta=1.0):
    cost_based_fscore = (1 + (alpha*beta)**2) * ((precision * recall) / ((alpha*beta)**2 * precision + recall))
    return cost_based_fscore
   ...custom function def cost_based_fscore() used to add metric cost_based_fscore
   ...successfully added custom metric
   ...generated image 2
   ...using scaled data for summarization

...generating mrm/mrm_section_10_template_descr_demo_scaled_out.html       